## Manual Review of LLM Scope/Pillar Classifications (S5)

Turns `scope_LLM` / `confidence_LLM` / `pillar_LLM` (from `S3_LLM_scope.py`) into a trusted `scope_curated` / `pillar_curated` verdict on `funding_classified`. LLM output alone isn't ground truth - confident decisions are auto-inherited, everything else goes to a human.

Standalone notebook, not invoked by `pipeline_funding.py` - run it whenever there's new LLM-scored data to review, independent of any single pipeline run.

Auto-accept logic (no ML stage in Funding, so this is a plain confidence-band split, unlike Publications' confidence+ML-pillar-agreement rule):
- Confidence 5-7 with a real pillar assigned -> auto-approve (`scope_curated='in'`)
- Confidence 1 with no pillar assigned -> auto-reject (`scope_curated='out'`) - "reject" means excluded from being promoted onward, **not** deleted from `funding_classified` (deleting it would break S1's re-query dedup check)
- Confidence 2-4 -> manual review
- Anything with a missing/inconsistent scope, pillar, or LLM status also falls through to manual review, regardless of confidence

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

DB_PATH = Path('../../Funding/funding.db')
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

today = datetime.today().strftime("%y%m%d")

## Pull LLM-scored grants for manual review

In [ ]:
# Connect to database
db = duckdb.connect(str(DB_PATH))

In [ ]:
# Show tables in database
db.sql("SHOW TABLES")

#### Get data from `funding_classified`

In [ ]:
data = db.sql("SELECT * FROM funding_classified").df()

In [ ]:
# Only rows that have actually been through S3 (LLM scoring)
data = data[data['status_LLM'].notna()]

In [ ]:
# funding_classified grows forever, so re-running this notebook must not re-export rows
# already curated in an earlier pass - only look at rows with no scope_curated yet.
if 'scope_curated' in data.columns:
    data = data[data['scope_curated'].isna()]

print(f"{len(data)} LLM-scored grants awaiting curation")

#### Create filter mask for curated scope and pillar

In [ ]:
# decision mask for auto-accept vs manual review
auto_in_mask = (
    (data['status_LLM'] == 'ok') &
    (data['scope_LLM'] == 'in') &
    (data['confidence_LLM'] >= 5) &
    (data['pillar_LLM'].notna()) & (data['pillar_LLM'] != 'NA')
)

auto_out_mask = (
    (data['status_LLM'] == 'ok') &
    (data['scope_LLM'] == 'out') &
    (data['confidence_LLM'] == 1) &
    (data['pillar_LLM'] == 'NA')
)

inherit_mask = auto_in_mask | auto_out_mask

#### Assign curated scope/pillar or manual review

In [ ]:
# Create curated scope and pillar columns with decision criteria
data['scope_curated']  = np.where(inherit_mask, data['scope_LLM'],  'manual_review')
data['pillar_curated'] = np.where(inherit_mask, data['pillar_LLM'], 'manual_review')
data['date_review'] = today

In [ ]:
# How many grants have to go through manual review?
data['scope_curated'].value_counts()

#### Save borderline grants for manual review

In [ ]:
# Export grants for review
review = data[data['scope_curated'] == 'manual_review']
review.to_csv(DATA_DIR / f'funding_for_review_{today}.csv', index=False)
print(f"Saved {len(review)} grants for review -> {DATA_DIR / f'funding_for_review_{today}.csv'}")

#### Save (automatically) curated scope and pillar back to `funding_classified`

In [ ]:
# filter for the grants with a clear scope and pillar assignment
data = data[(data['scope_curated'] != 'manual_review') & (~data['scope_curated'].isna())]

In [ ]:
# create columns in funding_classified if they don't already exist
db.sql("ALTER TABLE funding_classified ADD COLUMN IF NOT EXISTS scope_curated VARCHAR")
db.sql("ALTER TABLE funding_classified ADD COLUMN IF NOT EXISTS pillar_curated VARCHAR")

In [ ]:
# add to database
db.register('data', data[['id', 'scope_curated', 'pillar_curated']])
db.sql("""
    UPDATE funding_classified
    SET scope_curated  = data.scope_curated,
        pillar_curated = data.pillar_curated
    FROM data
    WHERE funding_classified.id = data.id
""")
print(f"Auto-curated {len(data)} grants in funding_classified")

In [ ]:
db.close()

## Assign manually curated scope and pillar to respective grants

Run this section after you've opened the exported CSV, filled in real `scope_curated` / `pillar_curated` values (replacing `manual_review`) for each row, and saved it as `funding_reviewed_{date}.csv` in the same `data/` folder.

In [ ]:
# Connect to database
db = duckdb.connect(str(DB_PATH))

In [ ]:
# load manually reviewed data - update the date in this filename to match your reviewed export
reviewed_data = pd.read_csv(DATA_DIR / 'funding_reviewed_YYMMDD.csv')

#### Add scope and pillar back to `funding_classified`, using grant ID

In [ ]:
# add to database
db.register('reviewed_data', reviewed_data[['id', 'scope_curated', 'pillar_curated']])
db.sql("""
    UPDATE funding_classified
    SET scope_curated  = reviewed_data.scope_curated,
        pillar_curated = reviewed_data.pillar_curated
    FROM reviewed_data
    WHERE funding_classified.id = reviewed_data.id
""")
print(f"Applied manual review decisions for {len(reviewed_data)} grants")

In [ ]:
data = db.sql("SELECT * FROM funding_classified").df()
data[~data['scope_curated'].isna()]

In [ ]:
db.close()